In [ ]:
#自动识别并读取 Excel、CSV、TXT 文件
#支持异常自动检测与提示。

In [20]:
folder_path = "D:/工作/特变电工/01_0政策与学习笔记/_全国_20250520/国网区域/新疆/400-交易复盘/现货日清分_系统导出数据/9月19日/"
filename="柯坪县柯特新能源有限责任公司2025-09-19-明细.xlsx"
file_path=os.path.join(folder_path, filename)

file_path
safe_read_table(file_path, encoding_list=('utf-8', 'gbk', 'ansi'))

D:\Develop\Anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,单位:MWh、元/MWh、元,NaN,NaN,NaN,NaN,NaN,NaN
1,结算单元,柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,合 计,NaN,NaN,NaN,NaN,NaN,NaN
2,时段,01:00,02:00,03:00,04:00,05:00,06:00,07:00,08:00,09:00,...,22:00,23:00,24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,上网电量,0,0,0,0,0,0,0,0,0,...,8.946,5.208,0,240.618,NaN,NaN,NaN,NaN,NaN,NaN
4,重庆市中海源电力有限责任公司与柯坪县柯特新能源有限责任公司（柯特新能源柯坪光伏一电站）202...,0,0,0,0,0,0,0,0,0,...,0,0,0,0.05,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,机组考核费用-分摊,0,0,0,0,0,0,0,0,0,...,0,0,0,175.28,NaN,NaN,NaN,NaN,NaN,NaN
230,用户侧现货偏差收益回收-分摊,/,/,/,/,/,/,/,/,/,...,/,/,/,10.47,NaN,NaN,NaN,NaN,NaN,NaN
231,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
import os
import pandas as pd
import zipfile
file_path
def safe_read_table(file_path, encoding_list=('utf-8', 'gbk', 'ansi')):
    """
    自动识别并读取 Excel、CSV、TXT 文件
    支持异常自动检测与提示。
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ 文件不存在: {file_path}")

    if os.path.getsize(file_path) == 0:
        raise ValueError(f"⚠️ 文件为空: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()

    # -------- Excel 文件处理 (.xlsx / .xlsm / .xls) --------
    if ext in ['.xlsx', '.xlsm', '.xls']:
        try:
            # 尝试用 openpyxl 读取新版 Excel
            return pd.read_excel(file_path, engine='openpyxl')
        except zipfile.BadZipFile:
            raise ValueError(f"⚠️ 文件后缀是 {ext}，但内容不是标准 Excel，请检查导出来源。")
        except Exception as e:
            # 如果是老格式 .xls，可尝试 xlrd
            if ext == '.xls':
                try:
                    return pd.read_excel(file_path, engine='xlrd')
                except Exception as e2:
                    raise ValueError(f"❌ 无法读取 .xls 文件，请确认文件是否损坏: {e2}")
            else:
                raise ValueError(f"❌ Excel 文件读取失败: {e}")

    # -------- CSV 文件处理 --------
    elif ext == '.csv':
        for enc in encoding_list:
            try:
                return pd.read_csv(file_path, encoding=enc)
            except UnicodeDecodeError:
                continue
        raise ValueError(f"❌ 无法识别 CSV 编码，请手动检查文件编码。")

    # -------- TXT 文件处理 --------
    elif ext == '.txt':
        for enc in encoding_list:
            try:
                return pd.read_csv(file_path, sep=None, engine='python', encoding=enc)
            except UnicodeDecodeError:
                continue
        raise ValueError(f"❌ 无法识别 TXT 编码，请手动检查文件编码。")

    # -------- 其他未知格式 --------
    else:
        # 尝试判断是不是伪装成 xlsx 的 CSV
        with open(file_path, 'rb') as f:
            head = f.read(200)
        if b',' in head or b'\t' in head:
            for enc in encoding_list:
                try:
                    return pd.read_csv(file_path, encoding=enc)
                except UnicodeDecodeError:
                    continue
        raise ValueError(f"⚠️ 不支持的文件类型或文件损坏: {file_path}")

